In [5]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import torch
from rank_bm25 import BM25Okapi
import faiss
from sentence_transformers import SentenceTransformer

# تعیین قطعی ریشه پروژه (یک پوشه بالاتر از notebook)
current_dir = Path.cwd().resolve()
if current_dir.name == "notebook":
    ROOT = current_dir.parent
else:
    ROOT = current_dir

print(f"PROJECT_ROOT = {ROOT}")

# بررسی و اضافه کردن src به مسیر ماژول‌ها
module_path = str(ROOT / "src")
if module_path not in sys.path:
    sys.path.insert(0, module_path)

from src.digikala_ai.utils.normalization import normalize_persian_text
print("✓ Normalization module imported successfully.")


PROJECT_ROOT = D:\goal\programming\quera\boot_camp\project3
✓ Normalization module imported successfully.


In [6]:
# Check processed first, fallback to raw if not found
processed_path = ROOT / "data" / "processed" / "products.parquet"
raw_path = ROOT / "data" / "raw" / "products.parquet"

if processed_path.exists():
    data_path = processed_path
    print(f"Loading cleaned products from: {data_path}")
elif raw_path.exists():
    data_path = raw_path
    print(f"Loading raw products from: {data_path}")
else:
    raise FileNotFoundError(f"Neither {processed_path} nor {raw_path} exists. Please check your data directory.")

df_products = pd.read_parquet(data_path)
print(f"Loaded products shape: {df_products.shape}")

# Select a clean, deterministic evaluation subset (2,000 items)
EVAL_CORPUS_SIZE = 2000
df_corpus = df_products.dropna(subset=["id", "title_fa"]).sample(
    n=EVAL_CORPUS_SIZE, random_state=42
).reset_index(drop=True)

print(f"✓ Shared evaluation corpus size: {len(df_corpus)}")


Loading cleaned products from: D:\goal\programming\quera\boot_camp\project3\data\processed\products.parquet
Loaded products shape: (1283496, 12)
✓ Shared evaluation corpus size: 2000


In [7]:
# Normalize titles and tokenize for BM25
df_corpus["title_norm"] = df_corpus["title_fa"].apply(normalize_persian_text)
corpus_tokens = df_corpus["title_norm"].apply(lambda x: x.split()).tolist()

print("Building BM25 index on shared corpus...")
bm25_index = BM25Okapi(corpus_tokens)
print("BM25 index built successfully.")


Building BM25 index on shared corpus...
BM25 index built successfully.


In [9]:
import torch
from sentence_transformers import SentenceTransformer

# 1. پیدا کردن مسیر کش محلی مدل در پوشه پروژه
model_cache_dir = ROOT / "artifacts" / "model_cache"
model_cache_repo_dir = model_cache_dir / "models--BAAI--bge-m3"
snapshots_dir = model_cache_repo_dir / "snapshots"

if not snapshots_dir.exists():
    raise FileNotFoundError(f"Model snapshots directory not found at: {snapshots_dir}")

# انتخاب جدیدترین snapshot محلی
snapshot_candidates = sorted(
    [path for path in snapshots_dir.iterdir() if path.is_dir() and (path / "config.json").exists()],
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

if not snapshot_candidates:
    raise FileNotFoundError(f"No valid model snapshots found in {snapshots_dir}")

local_model_path = snapshot_candidates[0]
print(f"Using local model snapshot: {local_model_path}")

# 2. لود مدل در حالت آفلاین روی پردازنده گرافیکی (CUDA)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

dense_model = SentenceTransformer(
    str(local_model_path),
    device=device,
    local_files_only=True
)

print("✓ Model loaded successfully from local cache!")
print(f"Embedding dimension: {dense_model.get_sentence_embedding_dimension()}")


Using local model snapshot: D:\goal\programming\quera\boot_camp\project3\artifacts\model_cache\models--BAAI--bge-m3\snapshots\5617a9f61b028005a4858fdac845db406aefb181
Using device: cuda
✓ Model loaded successfully from local cache!
Embedding dimension: 1024


In [10]:
import random

random.seed(42)
sample_queries_df = df_corpus.sample(n=20, random_state=42).reset_index(drop=True)

evaluation_queries = []
for idx, row in sample_queries_df.iterrows():
    target_id = row["id"]
    full_title = row["title_fa"]
    
    # Create realistic short query from the first 4 words of the title
    words = str(full_title).split()
    query_text = " ".join(words[:min(4, len(words))]) if len(words) >= 2 else full_title
    
    evaluation_queries.append({
        "id": f"QRY-{idx+1:02d}",
        "query": query_text,
        "target_id": target_id,
        "target_title": full_title
    })

queries_df = pd.DataFrame(evaluation_queries)
print(f"Generated {len(queries_df)} shared benchmark queries.")
queries_df.head()


Generated 20 shared benchmark queries.


,id,query,target_id,target_title
0,QRY-01,حلقه جاکلیدی کد br57,3412324,حلقه جاکلیدی کد br57 مجموعه 20 عددی
1,QRY-02,آویز گردنبند طلا 18,9725990,آویز گردنبند طلا 18 عیار دخترانه دوست خوب مدل ...
2,QRY-03,ژل شستشوی صورت درمدن,12077501,ژل شستشوی صورت درمدن مدل Purifying حجم 200 میل...
3,QRY-04,تی شرت آستین بلند,6565512,تی شرت آستین بلند نوزادی کارانس مدل TLB-3223
4,QRY-05,کتاب هفده گفتار در,5350591,کتاب هفده گفتار در علوم قرآنی اثر جمعی از نویس...


In [12]:
import numpy as np
import faiss

print("Preparing texts for embedding...")
# ترکیب عنوان و دسته‌بندی برای ساخت متن غنی‌تر برای جستجوی برداری
corpus_texts = (
    df_corpus["title_fa"].fillna("").astype(str) + " " + 
    df_corpus.get("category_title", pd.Series([""] * len(df_corpus))).fillna("").astype(str)
).tolist()

print(f"Encoding {len(corpus_texts)} corpus items with BGE-M3 (this may take a moment)...")
# استخراج امبدینگ‌ها با نرمال‌سازی جهت استفاده از Cosine Similarity
corpus_embeddings = dense_model.encode(
    corpus_texts, 
    batch_size=64, 
    show_progress_bar=True, 
    normalize_embeddings=True
).astype(np.float32)

embedding_dim = corpus_embeddings.shape[1]
print(f"Embedding shape: {corpus_embeddings.shape}")

# ساخت ایندکس FAISS
faiss_index = faiss.IndexFlatIP(embedding_dim)
faiss_index.add(corpus_embeddings)

print(f"✓ FAISS index built successfully with {faiss_index.ntotal} vectors!")


Preparing texts for embedding...
Encoding 2000 corpus items with BGE-M3 (this may take a moment)...


Batches: 100%|██████████| 32/32 [00:06<00:00,  4.65it/s]

Embedding shape: (2000, 1024)
✓ FAISS index built successfully with 2000 vectors!


In [13]:
bm25_results = []
dense_results = []

for _, q_row in queries_df.iterrows():
    query = q_row["query"]
    target_id = q_row["target_id"]
    
    # --- 1. BM25 Evaluation ---
    q_tokens = normalize_persian_text(query).split()
    bm25_scores = bm25_index.get_scores(q_tokens)
    bm25_sorted_idx = np.argsort(bm25_scores)[::-1][:10] # Top 10
    
    bm25_ranks = [i+1 for i, idx in enumerate(bm25_sorted_idx) if df_corpus.iloc[idx]["id"] == target_id]
    bm25_rank = bm25_ranks[0] if bm25_ranks else None
    bm25_results.append({"query": query, "rank": bm25_rank})
    
    # --- 2. Dense Evaluation ---
    q_emb = dense_model.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, indices = faiss_index.search(q_emb, 10) # Top 10
    
    dense_ranks = [i+1 for i, idx in enumerate(indices[0]) if idx != -1 and df_corpus.iloc[idx]["id"] == target_id]
    dense_rank = dense_ranks[0] if dense_ranks else None
    dense_results.append({"query": query, "rank": dense_rank})

# Compute metrics helper
def compute_metrics(res_list):
    hit1 = sum(1 for r in res_list if r["rank"] == 1)
    hit5 = sum(1 for r in res_list if r["rank"] is not None and r["rank"] <= 5)
    rr_sum = sum(1.0 / r["rank"] if r["rank"] is not None and r["rank"] > 0 else 0.0 for r in res_list)
    n = len(res_list)
    return {
        "Hit@1": hit1 / n,
        "Hit@5": hit5 / n,
        "MRR": rr_sum / n
    }

bm25_metrics = compute_metrics(bm25_results)
dense_metrics = compute_metrics(dense_results)

print("\n================ COMPARISON RESULTS ================")
print("BM25-only Metrics:")
for k, v in bm25_metrics.items():
    print(f"  {k}: {v:.4f}")

print("\nDense-only Metrics:")
for k, v in dense_metrics.items():
    print(f"  {k}: {v:.4f}")
print("====================================================")



================ COMPARISON RESULTS ================
BM25-only Metrics:
  Hit@1: 0.8000
  Hit@5: 0.9000
  MRR: 0.8338

Dense-only Metrics:
  Hit@1: 0.8500
  Hit@5: 0.8500
  MRR: 0.8500


In [14]:
import json
from datetime import datetime

# Check 15 percentage-point gate for BM25
bm25_hit5 = bm25_metrics["Hit@5"]
dense_hit5 = dense_metrics["Hit@5"]
gap = (dense_hit5 - bm25_hit5) * 100

print(f"Hit@5 Gap (Dense - BM25): {gap:.2f} percentage points")
if gap > 15:
    print("⚠️ BM25 Hit@5 is more than 15 percentage points lower than Dense. Recommendation: Exclude BM25 from RRF ensemble (Dense-only mode).")
else:
    print("✓ BM25 is within the acceptable threshold; eligible for Hybrid/RRF consideration in Day 4.")

# Save final Day 2 comparative report
report = {
    "timestamp": datetime.now().isoformat(),
    "corpus_size": EVAL_CORPUS_SIZE,
    "benchmark_queries_count": len(queries_df),
    "bm25_metrics": bm25_metrics,
    "dense_metrics": dense_metrics,
    "gate_analysis": {
        "hit5_gap_percentage_points": gap,
        "bm25_excluded_from_rrf": bool(gap > 15)
    }
}

report_path = ROOT / "evaluation" / "day2_retrieval_comparison_report.json"
report_path.parent.mkdir(parents=True, exist_ok=True)

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print(f"\n✓ Successfully saved Day 2 evaluation report to: {report_path}")


Hit@5 Gap (Dense - BM25): -5.00 percentage points
✓ BM25 is within the acceptable threshold; eligible for Hybrid/RRF consideration in Day 4.

✓ Successfully saved Day 2 evaluation report to: D:\goal\programming\quera\boot_camp\project3\evaluation\day2_retrieval_comparison_report.json


In [15]:
import json

report_path = ROOT / "evaluation" / "day2_retrieval_comparison_report.json"

if report_path.exists():
    with open(report_path, "r", encoding="utf-8") as f:
        report_data = json.load(f)
    
    print("=== گزارش ارزیابی روز دوم (Day 2 Evaluation Report) ===")
    print(json.dumps(report_data, indent=2, ensure_ascii=False))
else:
    print(f"File not found at {report_path}")


=== گزارش ارزیابی روز دوم (Day 2 Evaluation Report) ===
{
  "timestamp": "2026-11-23T16:35:12.654785",
  "corpus_size": 2000,
  "benchmark_queries_count": 20,
  "bm25_metrics": {
    "Hit@1": 0.8,
    "Hit@5": 0.9,
    "MRR": 0.8338095238095239
  },
  "dense_metrics": {
    "Hit@1": 0.85,
    "Hit@5": 0.85,
    "MRR": 0.85
  },
  "gate_analysis": {
    "hit5_gap_percentage_points": -5.000000000000004,
    "bm25_excluded_from_rrf": false
  }
}
